In [ ]:
# ===========================================================================
# Cell 0: setup. Paraphrase is cut; nothing below depends on it.
#
# Why it was cut, for the record: the pilot pass worked (556/600 kept, median token
# Jaccard 0.292 against originals) but the tag guard in make_paraphraser selected for
# degenerate rewrites. Forcing every mention to survive made the model bolt orphan tags
# onto trailing clauses ("...as noted for furosemide[1]"), and one rewrite kept all four
# tags while changing anakinra + TNF-blocker into anakinra + anakinra. The failures
# concentrate in "Entity: <statement>" headings, which is the construction the guidelines
# have a special rule for and where the annotator's errors already live, so the artefact
# would have inflated L's prune rate in the direction of the hypothesis. Fixing that
# needs a rebuilt guard and a re-validated pilot, and there is no time to validate it.
#
# The headline result does not depend on any of this: 0.785 / 0.657 / 0.462 at n=3,
# text provenance costing 0.195 against label provenance's 0.128.
# ===========================================================================
!pip install -q transformers torch scikit-learn openai tqdm pandas bioc spacy


In [ ]:

import os
import json
import importlib
import statistics
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from openai import OpenAI

import ddi.label_real, ddi.verify_binary, ddi.pruning, ddi.mixing, ddi.synth
for m in (ddi.label_real, ddi.verify_binary, ddi.pruning, ddi.mixing, ddi.synth):
    importlib.reload(m)

from ddi import label_real
from ddi.data import build_human, load_brat_docs, make_sentence_level, CORPUS
from ddi.manifest import load_dataset
from ddi.synth import generate_raw, RAW
from ddi.verify_binary import (build_batches, make_binary_verifier, load_verdicts,
                               calibration_report, group_by_sentence)
from ddi.pruning import build_arms
from ddi.mixing import subsample_by_sentence, match_negative_ratio, positive_rate
from ddi.train import train_and_eval
from ddi.experiment import log_run

MODEL, API = "gpt-oss-120b", "responses"
V18_ID = "20260825-095703-67c1fa"
LABEL_VERDICTS = RAW / "label-high-c8.jsonl"

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=180.0)

train, dev, val = build_human()
print(f"train {len(train)} pairs / {len(group_by_sentence(train))} sentences")
print(f"val   {len(val)} pairs / {len(group_by_sentence(val))} sentences")

label_inst, dropped = label_real.to_instances(train, LABEL_VERDICTS)
POOL_L = label_inst
POOL_H = [{**r, "label": r["gold_label"]} for r in label_inst]
POOL_S = load_dataset(V18_ID)[0]


In [ ]:

print(f"labelling arm: {len(POOL_L)} instances, "
      f"{len(group_by_sentence(POOL_L))} sentences, {dropped} dropped")
for n, p in (("H", POOL_H), ("L", POOL_L), ("S", POOL_S)):
    print(f"{n}: {len(p):>6} pairs, {len(group_by_sentence(p)):>5} sentences, "
          f"pos rate {positive_rate(p):.3f}")
print("corpus pair positive rate 0.162")



In [ ]:

# ===========================================================================
# Cell 1: does the Test directory parse.
#
# Ten minutes of work whose whole value is discovering the answer today rather than on
# Wednesday morning. load_brat_docs hardcodes CORPUS/<split>/{DrugBank,MedLine}/, and the
# official DDI-2013 release does not use that layout for test: it ships separate
# directories for the DrugNER and DDI-Extraction tasks. So the plain call is expected to
# fail, and what matters is what is actually on disk.
# ===========================================================================
root = Path(CORPUS)
print(f"{root.resolve()} exists: {root.exists()}\n")
for p in sorted(root.rglob("*")):
    if p.is_dir():
        n_ann = len(list(p.glob("*.ann")))
        n_txt = len(list(p.glob("*.txt")))
        if n_ann or n_txt:
            print(f"  {p.relative_to(root)}   {n_ann} ann, {n_txt} txt")

try:
    test_docs = load_brat_docs(split="Test")
    print(f"\nload_brat_docs('Test') worked: {len(test_docs)} documents")
except Exception as e:
    print(f"\nload_brat_docs('Test') failed: {type(e).__name__}: {e}")
    print("Expected. Use the loader below against whatever directories printed above.")


def load_test_docs(subdirs):
    """load_brat_docs with the directory list as an argument instead of hardcoded.

    Pass the directories that printed .ann counts above, relative to CORPUS, e.g.
    ["Test/Test for DDI Extraction task/DrugBank", "Test/.../MedLine"].
    """
    from bioc import brat
    docs = []
    for d in subdirs:
        dir_path = Path(CORPUS) / d
        for txt in sorted(dir_path.glob("*.txt")):
            ann = txt.with_suffix(".ann")
            if not ann.exists():
                continue
            with open(ann) as af, open(txt) as tf:
                doc = brat.load(tf, af)
            doc.register = "MedLine" if "medline" in d.lower() else "DrugBank"
            doc.doc_id = txt.stem
            docs.append(doc)
    return docs


TEST_DIRS = []          # fill in from the listing above
if TEST_DIRS:
    import spacy
    from ddi.data import make_pair_instances
    nlp = spacy.load("en_core_web_sm")
    test = [r for doc in load_test_docs(TEST_DIRS)
            for sent in make_sentence_level(doc, nlp)
            for r in make_pair_instances(sent)]
    print(f"test: {len(test)} pairs, {len(group_by_sentence(test))} sentences")
    print(Counter(r["label"] for r in test).most_common())
    print("\nreference: DDI-2013 test is ~5,700 pairs with ~980 positives. If the counts "
          "are far off, the directory list is wrong or the register split is uneven.")



In [ ]:

# ===========================================================================
# Cell 2: launch the three unattended API jobs, then leave them.
#
# All three are independent of each other and of the grid. Launch first, read later.
#
#   2a  verifier prune on v18. Never run on v18. Gained 0.041 on v14 and 0.042 on v15,
#       recall-only with precision flat, against a random-prune control.
#   2b  NONE-boundary prompt revision for the annotator. Its entire deficit is precision,
#       0.533 against human 0.749, with recall 0.857 already above human's 0.824.
#   2c  positives-only verifier pass over the existing annotator output. Same instrument,
#       calibrated at 0.948 positive recall, applied where the errors are.
#
# reasoning_effort stays high on both verifier and labeller. At low the five-way
# verifier's NONE recall collapsed to 0.45-0.48 and the v18 smoke run missed 21% of true
# positives against a calibrated 0.948. Every calibration number is a high-effort number.
# ===========================================================================

# 2a
generate_raw(build_batches(POOL_S), make_binary_verifier(client, model=MODEL),
             gen_id="v18-low-verify", max_workers=8)

# 2b. A revision, not a rewrite: keep the existing prompt and push on the boundary the
# error analysis actually implicates.
SYSTEM_V2 = label_real.SYSTEM.replace(
    "Answer for every pair, in the order given.",
    """Before assigning any positive label, check the pair against these, because the
guidelines treat them as NONE even when the sentence reads as though an interaction is
present. This is where labels are most often wrong.

A pair is NONE if the two mentions are the same substance, or a group and a member of
that group, however the sentence is worded.

A pair is NONE if the drugs are only co-administered, co-prescribed, listed, enumerated
or compared, and nothing is asserted about one changing the other. Being named in the
same clause is not an interaction.

A pair is NONE if the interaction is denied, reported as absent, described as not
clinically significant, or said not to have been studied or established.

A pair is NONE if the claim in the sentence is about one of these two drugs and some
third drug. Sitting in the same clause as the claim is not participating in it.

In a sentence of the form "Entity: <statement>", the leading entity participates only
if it is the subject of the statement. If the statement names its own two participants,
the heading mention is NONE against both of them.

Roughly five pairs in six in this corpus are NONE. If you are labelling many more than
one in six as positive, you are reading co-mention as interaction.

Answer for every pair, in the order given.""")
assert SYSTEM_V2 != label_real.SYSTEM, "prompt anchor not found; check label_real.SYSTEM"

_orig_system = label_real.SYSTEM
label_real.SYSTEM = SYSTEM_V2        
print(SYSTEM_V2)  # make_labeller reads it at call time


In [ ]:

specs_v2 = label_real.chunk(label_real.sample_sentences(train), size=8)
print(f"2b: {len(specs_v2)} chunked specs")
generate_raw(specs_v2, label_real.make_labeller(client, model=MODEL),
             gen_id="label-v2-c8", max_workers=32)
label_real.SYSTEM = _orig_system


In [ ]:

# 2c. Positives only. Everything the annotator called NONE is left alone, because the
# instrument's own false-positive rate on true NONEs is 8.0% and running it over 10,000
# correct negatives would generate more noise than signal.
pos_only = [r for r in POOL_L if r["label"] != "NONE"]
print(f"2c: {len(pos_only)} annotator positives over "
      f"{len(group_by_sentence(pos_only))} sentences")
generate_raw(build_batches(pos_only), make_binary_verifier(client, model=MODEL),
             gen_id="label-pos-verify", max_workers=8)


In [ ]:


# ===========================================================================
# Cell 3: the training helper. Everything below uses it.
#
# val, not dev. Every number to date is dev and dev has been iterated against since
# week 2. Val is untouched and this is a selection procedure, which is what val is for.
# Nothing in this notebook may touch test until cell 7.
# ===========================================================================
BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}

RESULTS = Path("reports/final_arms.jsonl")
RESULTS.parent.mkdir(parents=True, exist_ok=True)


def train_arm(name, records, eval_set=None, seeds=(0, 1, 2), notes="", extra=None):
    """One row appended to disk per seed, so a crashed grid is still partly usable."""
    eval_set = val if eval_set is None else eval_set
    rows = []
    for seed in seeds:
        cfg = {**BASE, "seed": seed, "dataset": name, **(extra or {})}
        m = train_and_eval(cfg, records, eval_set)
        log_run(cfg, m, notes=notes or name)
        row = {"arm": name, "seed": seed, "n": len(records),
               "pos_rate": round(positive_rate(records), 4),
               "f1": m["micro_f1_pos"], "p": m["micro_p_pos"], "r": m["micro_r_pos"],
               **(extra or {})}
        rows.append(row)
        with open(RESULTS, "a") as fp:
            fp.write(json.dumps(row) + "\n")
        print(f"  {name:<28} seed={seed} f1={row['f1']:.3f} "
              f"p={row['p']:.3f} r={row['r']:.3f}")
    df = pd.DataFrame(rows)
    print(f"  {name:<28} f1 {df.f1.mean():.3f} "
          f"(sd {df.f1.std():.3f})  P {df.p.mean():.3f}  R {df.r.mean():.3f}  n={len(df)}")
    return df



In [ ]:

# ===========================================================================
# Cell 4: 1b, the precision-fixed annotator. Standalone, two arms plus a filter arm.
#
# Deliberately decoupled from the grid. If the fixed annotator feeds the grid, the grid
# waits on it, and the grid is the larger deliverable. This reports as its own finding.
#
# Read the agreement numbers before training. The annotator scored 0.730 against gold
# untrained and the classifier trained on its labels reached 0.657, so -0.073 in the
# round trip. If agreement has not moved, do not spend GPU on the arm.
# ===========================================================================
print("=== original annotator ===")
label_real.agreement(LABEL_VERDICTS)
print("\n=== revised annotator ===")
label_real.agreement(RAW / "label-v2-c8.jsonl")

label_v2, dropped_v2 = label_real.to_instances(train, RAW / "label-v2-c8.jsonl")
print(f"\nrevised: {len(label_v2)} instances, {dropped_v2} dropped, "
      f"pos rate {positive_rate(label_v2):.3f} (original {positive_rate(POOL_L):.3f}, "
      f"corpus 0.162)")

# 2c's filter arm: drop annotator positives the verifier judged not annotated.
vpos = load_verdicts("label-pos-verify")
flagged_off = {(v.sent_id, v.m1, v.m2) for v in vpos.itertuples() if not v.flagged}
from ddi.pruning import instance_keys
_keys, _ = instance_keys(POOL_L)
_ordered = [r for sid in group_by_sentence(POOL_L) for r in group_by_sentence(POOL_L)[sid]]
label_filtered = [{**r, "label": "NONE"} if k in flagged_off else r
                  for r, k in zip(_ordered, _keys)]
print(f"filter arm: {len(flagged_off)} positives demoted to NONE, "
      f"pos rate {positive_rate(label_filtered):.3f}")

res_1b = [
    train_arm("L", POOL_L, notes="labelling arm, original prompt"),
    train_arm("L-v2", label_v2, notes="labelling arm, NONE-boundary revision"),
    train_arm("L-filtered", label_filtered, notes="labelling arm, verifier on positives"),
    train_arm("H", POOL_H, notes="human ceiling, matched instances"),
]
print("""
reference, dev, not directly comparable to these val numbers:
  human 0.785  P 0.749  R 0.824      LLM labels 0.657  P 0.533  R 0.857

  L-v2 gains precision, recall roughly flat   -> the deficit was conventions, the fix
     landed, and the headline moves toward "labelling nearly matches human supervision"
  L-v2 gains precision and loses recall       -> the prompt made it conservative rather
     than accurate. Report both and do not claim the boundary was learned.
  L-filtered beats L-v2                       -> the instrument is a better fix than the
     prompt, which is a cheaper and more transferable finding
""")


In [ ]:


# ===========================================================================
# Cell 5: v18 pruning arms. Second standalone comparison, no grid dependency.
#
# The random-prune control is what makes it interpretable: the verifier flags roughly the
# same fraction of NONEs as its own false-positive rate, so most flags are the verifier
# being wrong, and if random removal does the same thing the verifier contributed nothing.
# ===========================================================================
vdf_s = load_verdicts("v18-low-verify")
calibration_report(vdf_s)
arms_s = build_arms(POOL_S, vdf_s, seed=0)
for n, recs in arms_s.items():
    print(f"{n:<20} {len(recs):>6} instances  pos rate {positive_rate(recs):.3f}")

res_prune = [train_arm(f"v18-{n}", recs, notes="v18 pruning arms, val")
             for n, recs in arms_s.items()]
print("""
  judged-pruned above judged AND above judged-randprune -> replicates on a third
     generator. Recall-only was the mechanism on v14 and v15; check P and R separately.
  judged-pruned matches judged-randprune                -> quantity, not the verifier.
     That is a clean negative and it retracts a claim the literature assessment called
     the most novel contribution, so state it plainly.
""")



In [ ]:

# ===========================================================================
# Cell 6: the grid on val. Launch before bed.
#
# Human budget crossed with what you add to it. Budgets cut to four; the interesting
# cells are the ends and the 250 and 500 rows were the first thing the plan said to drop.
#
# Five seeds below 500 sentences, three above, following mixing.py: at budget 100 the
# human-only arm has an sd of 0.105 and three seeds cannot carry it.
#
# The positive-rate control is not optional. LLM labels run 0.242 positive, synthetic
# 0.157, corpus 0.162. A mixed arm sits between them and can win on class balance rather
# than on content, and without the control you cannot say which.
# ===========================================================================
BUDGETS = [0, 100, 1000, 2248]
ADDITIONS = {"none": [], "synth": POOL_S, "llm": POOL_L, "both": POOL_S + POOL_L}
TARGET_POS = 0.162

grid = []
for budget in BUDGETS:
    seeds = (0, 1, 2, 3, 4) if budget < 500 else (0, 1, 2)
    for add_name, add in ADDITIONS.items():
        if budget == 0 and add_name == "none":
            continue                                  # empty training set
        for seed in seeds:
            sub = subsample_by_sentence(POOL_H, budget, seed) if budget else []
            used = {r["sent_id"] for r in sub}
            rest = [r for r in POOL_L if r["sent_id"] not in used]   # LLM labels only where no human label exists
            add = {"none": [], "synth": POOL_S, "llm": rest, "both": POOL_S + rest}[add_name]
            data = sub + add
            grid.append((f"grid-b{budget}-{add_name}", data, seed,
                         {"budget_sentences": budget, "addition": add_name,
                          "match_balance": False}))
            # control at the two ends only
            if add_name in ("llm", "both") and budget in (100, 2248):
                grid.append((f"grid-b{budget}-{add_name}-bal",
                             match_negative_ratio(data, TARGET_POS, seed), seed,
                             {"budget_sentences": budget, "addition": add_name,
                              "match_balance": True}))

print(f"{len(grid)} runs, {len({g[0] for g in grid})} cells")
print("smoke-test one cell before launching the rest:")
smoke = grid[0]
train_arm(smoke[0], smoke[1], seeds=(smoke[2],), notes="grid smoke test", extra=smoke[3])

# ---------------------------------------------------------------------------
# Only after the smoke test prints a plausible number. Resumable: cells already in
# final_arms.jsonl are skipped, so a dead pod costs one run.
# ---------------------------------------------------------------------------
done = set()
if RESULTS.exists():
    for line in RESULTS.read_text().splitlines():
        if line:
            r = json.loads(line)
            done.add((r["arm"], r["seed"]))

for name, data, seed, extra in grid:
    if (name, seed) in done:
        continue
    train_arm(name, data, seeds=(seed,), notes="val grid", extra=extra)


In [ ]:


# ===========================================================================
# Cell 7: read the grid, choose on val, then run test once.
#
# Write the selection to disk before the test run. That timestamp is the only thing
# standing between you and re-selecting after seeing the test number, which is trivially
# easy to do by accident and is the only thing making a test number worth more than a
# dev number.
# ===========================================================================
rows = [json.loads(l) for l in RESULTS.read_text().splitlines() if l]
g = pd.DataFrame([r for r in rows if r["arm"].startswith("grid-")])
tab = (g.groupby(["budget_sentences", "addition", "match_balance"])
        [["f1", "p", "r"]].agg(["mean", "std", "count"]))
print(tab.to_string())

best = (g[~g.match_balance].groupby(["budget_sentences", "addition"])["f1"]
        .mean().reset_index())
print("\nbest addition at each budget")
for b, sub in best.groupby("budget_sentences"):
    top = sub.sort_values("f1", ascending=False).iloc[0]
    print(f"  budget {b:>5}: {top.addition:<6} {top.f1:.3f}   " +
          "  ".join(f"{r.addition} {r.f1:.3f}" for r in sub.itertuples()))

SELECTION = {"budget": None, "addition": None, "reason": ""}   # fill in from the above
assert SELECTION["addition"], "choose on val and write it down before running test"

sel_path = Path("reports/test_selection.json")
sel_path.write_text(json.dumps(
    {**SELECTION, "chosen_at": pd.Timestamp.now().isoformat(),
     "val_table": tab.reset_index().to_dict("records")}, indent=1))
print(f"\nselection frozen -> {sel_path}. Commit this before the next cell.")


In [ ]:

# ---------------------------------------------------------------------------
# Test. Three arms, five seeds, roughly half an hour. Run once.
# ---------------------------------------------------------------------------
sel_add = ADDITIONS[SELECTION["addition"]]
sel_data = subsample_by_sentence(POOL_H, SELECTION["budget"], 0) + list(sel_add) \
    if SELECTION["budget"] else list(sel_add)

res_test = [
    train_arm("test-chosen", sel_data, eval_set=test, seeds=(0, 1, 2, 3, 4),
              notes=f"TEST, val-chosen: budget={SELECTION['budget']} "
                    f"addition={SELECTION['addition']}"),
    train_arm("test-human-ceiling",
              subsample_by_sentence(POOL_H, SELECTION["budget"] or 2248, 0),
              eval_set=test, seeds=(0, 1, 2, 3, 4), notes="TEST, human ceiling"),
    train_arm("test-synth-floor", POOL_S, eval_set=test, seeds=(0, 1, 2, 3, 4),
              notes="TEST, synthetic-only floor"),
]
print("""
Report what val chose, whatever test says. If a different configuration would have won on
test, that goes in the writeup as a sentence, not as a change of arm.
""")

In [ ]:
# ===========================================================================
# Cell 5b: v18 pruning, two more seeds and the question of what was removed.
#
# At three seeds pruned beats random-prune by 0.026 at 1.8 standard errors, with the
# same recall-only signature as v14 and v15. An effect of that size is exactly what the
# v18 notebook said needs five seeds to carry, so seeds 3 and 4 go on the two arms that
# the comparison rests on. The unpruned judged arm is not rerun; its sd is 0.006.
#
# The second question is whether the gain comes from undoing v18's own design. The
# `unlisted` variant deliberately writes sentences whose surface reads positive and
# whose gold is NONE, and the verifier flags exactly that shape. If pruning removes
# unlisted sentences at a much higher rate than plain ones, some of the recall gain is
# the verifier deleting the hard negatives v18 was built to add.
#
# Run in the same kernel as cells 0 and 3. If the kernel has restarted, run those first;
# the guard below rebuilds the arms, and build_arms with seed=0 is deterministic, so the
# random-prune arm is the same one the first three seeds trained on.
# ===========================================================================
import re
import json
import statistics as st
from collections import Counter, defaultdict

if "arms_s" not in globals():
    vdf_s = load_verdicts("v18-low-verify")
    arms_s = build_arms(POOL_S, vdf_s, seed=0)
    print("rebuilt arms:", {k: len(v) for k, v in arms_s.items()})

for name in ["judged-pruned", "judged-randprune"]:
    train_arm(f"v18-{name}", arms_s[name], seeds=(3, 4), notes="v18 pruning arms, val")

# ---------------------------------------------------------------------------
# Summary across every seed on disk, with the standard error of each difference.
# ---------------------------------------------------------------------------
rows = [json.loads(l) for l in RESULTS.read_text().splitlines() if l]
by = defaultdict(list)
for r in rows:
    if r["arm"] in ("v18-judged", "v18-judged-pruned", "v18-judged-randprune"):
        by[r["arm"]].append(r)

summ = {}
print(f"\n{'arm':<22} {'n':>2} {'F1':>6} {'sd':>6} {'P':>6} {'R':>6}")
for arm in ("v18-judged", "v18-judged-pruned", "v18-judged-randprune"):
    rs = by[arm]
    f = [r["f1"] for r in rs]
    summ[arm] = (st.mean(f), st.stdev(f) if len(f) > 1 else 0.0, len(f))
    print(f"{arm:<22} {len(f):>2} {st.mean(f):>6.3f} {summ[arm][1]:>6.3f} "
          f"{st.mean(r['p'] for r in rs):>6.3f} {st.mean(r['r'] for r in rs):>6.3f}")

def diff(a, b):
    (ma, sa, na), (mb, sb, nb) = summ[a], summ[b]
    se = (sa ** 2 / na + sb ** 2 / nb) ** 0.5
    return ma - mb, se, (ma - mb) / se if se else float("inf")

for a, b in [("v18-judged-pruned", "v18-judged-randprune"),
             ("v18-judged-pruned", "v18-judged")]:
    d, se, z = diff(a, b)
    print(f"\n{a} minus {b}: {d:+.3f}  (se {se:.3f}, {z:.1f} se)")

print("""
reference: v14 pruned beat random-prune by +0.041, v15 by +0.042, both recall-only.
  above ~2 se with recall carrying it   -> replicates on a third generator
  below ~2 se                           -> direction replicates, magnitude does not
                                           resolve at five seeds; report it that way
These are val numbers. Do not compare them with the 0.486 v18 dev figure.
""")

# ---------------------------------------------------------------------------
# What pruning removed, by v18 variant.
#
# The removed set is the judged arm minus the pruned arm. Each removed instance is
# traced to the spec that produced it through the sent_id join, and its variant is read
# from the raw generation record.
# ---------------------------------------------------------------------------
SUF = re.compile(r":s\d+$")
gen = POOL_S[0]["sent_id"].split(":")[1]
variant = {}
for line in (RAW / f"{gen}.jsonl").read_text().splitlines():
    if not line:
        continue
    r = json.loads(line)
    if r.get("sample") and not r.get("error"):
        v = r["spec"].get("variants") or ()
        variant[f"synth:{gen}:{r['spec_index']}"] = "+".join(v) if v else "plain"

key = lambda r: (r["sent_id"], r["text"])
kept = {key(r) for r in arms_s["judged-pruned"]}
judged = arms_s["judged"]
removed = [r for r in judged if key(r) not in kept]
var_of = lambda r: variant.get(SUF.sub("", r["sent_id"]), "?")

all_v, rem_v = Counter(map(var_of, judged)), Counter(map(var_of, removed))
print(f"{len(removed)} instances removed of {len(judged)} "
      f"({len(removed) / len(judged):.1%}), from generation {gen}\n")
print(f"{'variant':<24} {'judged':>8} {'removed':>8} {'removal rate':>13}")
for v, n in all_v.most_common():
    print(f"{v:<24} {n:>8} {rem_v[v]:>8} {rem_v[v] / n:>13.3%}")

print("\nremoved instances by gold label:", dict(Counter(r["label"] for r in removed)))
print("""
  unlisted removed at a rate well above plain -> the verifier is deleting v18's
     deliberate hard negatives, and part of the gain undoes v18's design. Say so.
  rates roughly equal across variants          -> pruning is not targeting the
     variants, and the gain is not coming from undoing them.
""")

In [ ]:
import random
unl = [r for r in removed if "unlisted" in var_of(r)]
for r in random.Random(0).sample(unl, 20):
    print(r["text"], "\n")